In [0]:
from pyspark.sql import functions as F

silver_loads = spark.table(
    "workspace.transportation_analytics.silver_loads"
)

silver_routes = spark.table(
    "workspace.transportation_analytics.silver_routes"
)

In [0]:
silver_loads.printSchema()
silver_routes.printSchema()

root
 |-- load_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- weight_lbs: integer (nullable = true)
 |-- pieces: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- fuel_surcharge: double (nullable = true)
 |-- accessorial_charges: integer (nullable = true)
 |-- load_status: string (nullable = true)
 |-- booking_type: string (nullable = true)

root
 |-- route_id: string (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- typical_distance_miles: integer (nullable = true)
 |-- base_rate_per_mile: double (nullable = true)
 |-- fuel_surcharge_rate: double (nullable = true)
 |-- typical_transit_days: integer (nullable = true)



In [0]:
monthly_loads = (
    silver_loads
    .withColumn(
        "month",
        F.date_trunc("month", "load_date")
    )
    .groupBy("month")
    .agg(
        F.countDistinct("load_id").alias("total_loads")
    )
    .orderBy("month")
)

display(monthly_loads)

month,total_loads
2022-01-01T00:00:00.000Z,2466
2022-02-01T00:00:00.000Z,2179
2022-03-01T00:00:00.000Z,2370
2022-04-01T00:00:00.000Z,2346
2022-05-01T00:00:00.000Z,2428
2022-06-01T00:00:00.000Z,2376
2022-07-01T00:00:00.000Z,2421
2022-08-01T00:00:00.000Z,2496
2022-09-01T00:00:00.000Z,2304
2022-10-01T00:00:00.000Z,2468


In [0]:
monthly_revenue = (
    silver_loads
    .withColumn(
        "month",
        F.date_trunc("month", "load_date")
    )
    .groupBy("month")
    .agg(
        F.sum("revenue").alias("total_revenue")
    )
    .orderBy("month")
)

display(monthly_revenue)

month,total_revenue
2022-01-01T00:00:00.000Z,7566361.990000011
2022-02-01T00:00:00.000Z,6574259.059999994
2022-03-01T00:00:00.000Z,7401197.04999999
2022-04-01T00:00:00.000Z,7156041.220000005
2022-05-01T00:00:00.000Z,7518061.329999987
2022-06-01T00:00:00.000Z,7325637.689999998
2022-07-01T00:00:00.000Z,7380097.230000001
2022-08-01T00:00:00.000Z,7716839.569999993
2022-09-01T00:00:00.000Z,7108889.830000004
2022-10-01T00:00:00.000Z,7501831.5500000045


In [0]:
freight_rate = (
    silver_loads
    .join(
        silver_routes.select(
            "route_id",
            "typical_distance_miles"
        ),
        on="route_id",
        how="left"
    )
    .withColumn(
        "month",
        F.date_trunc("month", "load_date")
    )
    .groupBy("month")
    .agg(
        F.round(
            F.sum("revenue") /
            F.sum("typical_distance_miles"),
            2
        ).alias("freight_rate_per_mile")
    )
    .orderBy("month")
)

display(freight_rate)

month,freight_rate_per_mile
2022-01-01T00:00:00.000Z,2.2
2022-02-01T00:00:00.000Z,2.2
2022-03-01T00:00:00.000Z,2.21
2022-04-01T00:00:00.000Z,2.22
2022-05-01T00:00:00.000Z,2.23
2022-06-01T00:00:00.000Z,2.2
2022-07-01T00:00:00.000Z,2.19
2022-08-01T00:00:00.000Z,2.21
2022-09-01T00:00:00.000Z,2.22
2022-10-01T00:00:00.000Z,2.21


In [0]:
gold_seasonal_patterns = (
    monthly_loads
    .join(monthly_revenue, "month", "left")
    .join(freight_rate, "month", "left")
    .orderBy("month")
)

display(gold_seasonal_patterns)

month,total_loads,total_revenue,freight_rate_per_mile
2022-01-01T00:00:00.000Z,2466,7566361.990000011,2.2
2022-02-01T00:00:00.000Z,2179,6574259.059999994,2.2
2022-03-01T00:00:00.000Z,2370,7401197.04999999,2.21
2022-04-01T00:00:00.000Z,2346,7156041.220000005,2.22
2022-05-01T00:00:00.000Z,2428,7518061.329999987,2.23
2022-06-01T00:00:00.000Z,2376,7325637.689999998,2.2
2022-07-01T00:00:00.000Z,2421,7380097.230000001,2.19
2022-08-01T00:00:00.000Z,2496,7716839.569999993,2.21
2022-09-01T00:00:00.000Z,2304,7108889.830000004,2.22
2022-10-01T00:00:00.000Z,2468,7501831.5500000045,2.21


In [0]:
gold_seasonal_patterns.limit(0).write \
    .format("delta") \
    .saveAsTable(
        "workspace.transportation_analytics.gold_seasonal_patterns"
    )

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_seasonal_patterns"
)
target.alias("t").merge(
    gold_seasonal_patterns.alias("s"),
    "t.month = s.month"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_seasonal_patterns"
    )
)

month,total_loads,total_revenue,freight_rate_per_mile
2022-01-01T00:00:00.000Z,2466,7566361.990000011,2.2
2022-02-01T00:00:00.000Z,2179,6574259.059999994,2.2
2022-03-01T00:00:00.000Z,2370,7401197.04999999,2.21
2022-04-01T00:00:00.000Z,2346,7156041.220000005,2.22
2022-05-01T00:00:00.000Z,2428,7518061.329999987,2.23
2022-06-01T00:00:00.000Z,2376,7325637.689999998,2.2
2022-07-01T00:00:00.000Z,2421,7380097.230000001,2.19
2022-08-01T00:00:00.000Z,2496,7716839.569999993,2.21
2022-09-01T00:00:00.000Z,2304,7108889.830000004,2.22
2022-10-01T00:00:00.000Z,2468,7501831.5500000045,2.21


In [0]:
display(
    gold_seasonal_patterns.select(
        "month",
        "total_loads"
    ).orderBy("month")
)

month,total_loads
2022-01-01T00:00:00.000Z,2466
2022-02-01T00:00:00.000Z,2179
2022-03-01T00:00:00.000Z,2370
2022-04-01T00:00:00.000Z,2346
2022-05-01T00:00:00.000Z,2428
2022-06-01T00:00:00.000Z,2376
2022-07-01T00:00:00.000Z,2421
2022-08-01T00:00:00.000Z,2496
2022-09-01T00:00:00.000Z,2304
2022-10-01T00:00:00.000Z,2468


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_seasonal_patterns.select(
        "month",
        "total_revenue"
    ).orderBy("month")
)

month,total_revenue
2022-01-01T00:00:00.000Z,7566361.990000011
2022-02-01T00:00:00.000Z,6574259.059999994
2022-03-01T00:00:00.000Z,7401197.04999999
2022-04-01T00:00:00.000Z,7156041.220000005
2022-05-01T00:00:00.000Z,7518061.329999987
2022-06-01T00:00:00.000Z,7325637.689999998
2022-07-01T00:00:00.000Z,7380097.230000001
2022-08-01T00:00:00.000Z,7716839.569999993
2022-09-01T00:00:00.000Z,7108889.830000004
2022-10-01T00:00:00.000Z,7501831.5500000045


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_seasonal_patterns.select(
        "month",
        "freight_rate_per_mile"
    ).orderBy("month")
)

month,freight_rate_per_mile
2022-01-01T00:00:00.000Z,2.2
2022-02-01T00:00:00.000Z,2.2
2022-03-01T00:00:00.000Z,2.21
2022-04-01T00:00:00.000Z,2.22
2022-05-01T00:00:00.000Z,2.23
2022-06-01T00:00:00.000Z,2.2
2022-07-01T00:00:00.000Z,2.19
2022-08-01T00:00:00.000Z,2.21
2022-09-01T00:00:00.000Z,2.22
2022-10-01T00:00:00.000Z,2.21


Databricks visualization. Run in Databricks to view.

In [0]:
gold_seasonal_patterns = gold_seasonal_patterns.withColumn(
    "season",
    F.when(F.month("month").isin(12, 1, 2), "Winter")
     .when(F.month("month").isin(3, 4, 5), "Spring")
     .when(F.month("month").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
)

display(gold_seasonal_patterns)

month,total_loads,total_revenue,freight_rate_per_mile,season
2022-01-01T00:00:00.000Z,2466,7566361.990000011,2.2,Winter
2022-02-01T00:00:00.000Z,2179,6574259.059999994,2.2,Winter
2022-03-01T00:00:00.000Z,2370,7401197.04999999,2.21,Spring
2022-04-01T00:00:00.000Z,2346,7156041.220000005,2.22,Spring
2022-05-01T00:00:00.000Z,2428,7518061.329999987,2.23,Spring
2022-06-01T00:00:00.000Z,2376,7325637.689999998,2.2,Summer
2022-07-01T00:00:00.000Z,2421,7380097.230000001,2.19,Summer
2022-08-01T00:00:00.000Z,2496,7716839.569999993,2.21,Summer
2022-09-01T00:00:00.000Z,2304,7108889.830000004,2.22,Autumn
2022-10-01T00:00:00.000Z,2468,7501831.5500000045,2.21,Autumn


In [0]:
display(
    gold_seasonal_patterns
    .groupBy("season")
    .agg(
        F.sum("total_loads").alias("total_loads")
    )
    .orderBy("season")
)

season,total_loads
Autumn,21390
Spring,21467
Summer,21620
Winter,20933


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_seasonal_patterns
    .groupBy("season")
    .agg(
        F.avg("freight_rate_per_mile").alias(
            "avg_freight_rate_per_mile"
        )
    )
    .orderBy("season")
)

season,avg_freight_rate_per_mile
Autumn,2.2111111111111112
Spring,2.2144444444444447
Summer,2.2133333333333334
Winter,2.21


Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql("""
ALTER TABLE workspace.transportation_analytics.gold_seasonal_patterns
ADD COLUMNS (season STRING)
""")

DataFrame[]

In [0]:
print(gold_seasonal_patterns.columns)

['month', 'total_loads', 'total_revenue', 'freight_rate_per_mile', 'season']


In [0]:
target.alias("t").merge(
    gold_seasonal_patterns.alias("s"),
    "t.month = s.month"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_seasonal_patterns"
    ).select(
        "month",
        "season",
        "total_loads",
        "freight_rate_per_mile"
    ).orderBy("month")
)

month,season,total_loads,freight_rate_per_mile
2022-01-01T00:00:00.000Z,Winter,2466,2.2
2022-02-01T00:00:00.000Z,Winter,2179,2.2
2022-03-01T00:00:00.000Z,Spring,2370,2.21
2022-04-01T00:00:00.000Z,Spring,2346,2.22
2022-05-01T00:00:00.000Z,Spring,2428,2.23
2022-06-01T00:00:00.000Z,Summer,2376,2.2
2022-07-01T00:00:00.000Z,Summer,2421,2.19
2022-08-01T00:00:00.000Z,Summer,2496,2.21
2022-09-01T00:00:00.000Z,Autumn,2304,2.22
2022-10-01T00:00:00.000Z,Autumn,2468,2.21
